In [2]:
#TASK 1
students = ["Ali", "Sara", "Ahmed", "Zoya", "Bilal", "Hina"]

#domains
domains = {
    "Ali":    [1, 2, 3, 4, 5, 6],
    "Sara":   [1, 2, 3, 4, 5, 6],
    "Ahmed":  [1, 2, 3, 4, 5, 6],
    "Zoya":   [1, 2, 3, 4, 5, 6],
    "Bilal":  [2, 4, 6],           #even seats only
    "Hina":   [2, 3, 4, 5, 6]      #cannot sit on seat 1
}

def is_valid(assignment, student, seat):

    for s in assignment:
        if assignment[s] == seat:
            return False

    assignment[student] = seat


    if "Sara" in assignment and "Zoya" in assignment:
        if abs(assignment["Sara"] - assignment["Zoya"]) != 1:
            del assignment[student]
            return False

    if "Ali" in assignment and "Ahmed" in assignment:
        if abs(assignment["Ali"] - assignment["Ahmed"]) == 1:
            del assignment[student]
            return False

    del assignment[student]
    return True


def backtrack(assignment):
    if len(assignment) == len(students):
        return assignment

    for student in students:
        if student not in assignment:
            break

    for seat in domains[student]:
        if is_valid(assignment, student, seat):
            assignment[student] = seat

            result = backtrack(assignment)
            if result:
                return result

            #Backtrack
            del assignment[student]

    return None

solution = backtrack({})

if solution:
    print("Seating Arrangement:")
    for student in solution:
        print(student, "-> Seat", solution[student])
else:
    print("No solution found.")

Seating Arrangement:
Ali -> Seat 1
Sara -> Seat 2
Ahmed -> Seat 4
Zoya -> Seat 3
Bilal -> Seat 6
Hina -> Seat 5


In [1]:
#TASK 3
from ortools.sat.python import cp_model

def solve_course_timetabling():
    model = cp_model.CpModel()

    courses = ["CS101", "CS102", "AI201", "DS202", "SE301", "ML401"]
    rooms = ["R1", "R2", "R3"]

    room_capacity = {"R1": 40, "R2": 60, "R3": 100}

    students = {
        "CS101": 35,
        "CS102": 50,
        "AI201": 45,
        "DS202": 60,
        "SE301": 30,
        "ML401": 80
    }

    conflicts = [
        ("CS101", "CS102"),
        ("CS102", "AI201"),
        ("AI201", "ML401"),
        ("DS202", "SE301")
    ]

    timeslot = {}
    room = {}

    for c in courses:
        timeslot[c] = model.NewIntVar(1, 4, f"time_{c}")
        room[c] = model.NewIntVar(0, 2, f"room_{c}")

    for c1, c2 in conflicts:
        model.Add(timeslot[c1] != timeslot[c2])

    for c in courses:
        allowed = []
        for i, r in enumerate(rooms):
            if room_capacity[r] >= students[c]:
                allowed.append([i])
        model.AddAllowedAssignments([room[c]], allowed)

    for i in range(len(courses)):
        for j in range(i + 1, len(courses)):
            c1, c2 = courses[i], courses[j]

            same_time = model.NewBoolVar(f"same_{c1}_{c2}")

            model.Add(timeslot[c1] == timeslot[c2]).OnlyEnforceIf(same_time)
            model.Add(timeslot[c1] != timeslot[c2]).OnlyEnforceIf(same_time.Not())

            model.Add(room[c1] != room[c2]).OnlyEnforceIf(same_time)

    model.Add(timeslot["ML401"] >= 3)
    model.Add(timeslot["DS202"] <= 2)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
        print("Timetable:\n")
        room_names = ["R1", "R2", "R3"]

        for c in courses:
            print(f"{c}: Time {solver.Value(timeslot[c])}, Room {room_names[solver.Value(room[c])]}")
    else:
        print("No solution found")


solve_course_timetabling()

Timetable:

CS101: Time 4, Room R3
CS102: Time 3, Room R2
AI201: Time 4, Room R2
DS202: Time 1, Room R2
SE301: Time 2, Room R3
ML401: Time 3, Room R3
